### You are going to classify 5 types of flowers, i.e: tulips, sunflowers, roses, dandelions and daisies. You will try to apply convolutional neural network architecture (CNN) and transfer learning. Think, which one of them could be better and why likewise memorize the details what is the receipt of writting a code.


Firstly, you have to upload the zip file with the data

In [ ]:
# Dataset available locally

Unzip the dataset which contains the flower dataset

In [ ]:
import os
classes = sorted([d for d in os.listdir('./flower_photos') if os.path.isdir(os.path.join('./flower_photos', d))])
print(f"Classes: {classes}")


### Prepare datasets

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os


dataset_path = './flower_photos'

# Prepare train and test image transforms
train_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((256, 256)),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((256, 256)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create dataset instances
train_dataset_full = ImageFolder(dataset_path, transform=train_transform)
test_dataset_full = ImageFolder(dataset_path, transform=test_transform)

# Train and test dataset split
indices = np.arange(len(train_dataset_full))
train_set_indices, test_set_indices = train_test_split(indices, test_size=0.2)
train_dataset = Subset(train_dataset_full, train_set_indices)
test_dataset = Subset(test_dataset_full, test_set_indices)

num_classes = len(train_dataset_full.classes)

print(f"Train datset size: {len(train_dataset)}")
print(f"Test datset size: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

# ----------------------------------------------------------------------------------------------------------------------
# Plot the first 9 images from the train set
fig, axes = plt.subplots(3, 3, figsize=(6, 6))
fig.suptitle("Flowers dataset - training samples", fontsize=14)

for i in range(9):
    image, label = train_dataset[i]
    image = image.squeeze().permute(1, 2, 0).numpy()

    # Normalize to 0-1 range (this is optional and offers better visualization, pyplot expects the 0-1 range for float dtypes)
    image = image - np.min(image)
    image = image / np.max(image)

    ax = axes[i // 3, i % 3]
    ax.imshow(image)
    ax.set_title(f"Label: {label}")
    ax.axis('off')

plt.tight_layout()
plt.show()
# ----------------------------------------------------------------------------------------------------------------------

# Create dataloader instances
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Function for model evaluation on the test subset.
def test(model, loss_func, test_loader, device):
    model.eval()
    test_losses = []
    test_preds = []
    test_gt = []

    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        logits = model(inputs)
        loss = loss_func(logits, labels)
        test_losses.append(loss.item())

        pred = torch.argmax(logits, dim=1)
        test_preds.extend(pred.cpu().numpy())
        test_gt.extend(labels.detach().cpu().numpy())

    test_loss = sum(test_losses) / len(test_losses)
    test_acc = accuracy_score(test_gt, test_preds)

    model.train()

    return test_loss, test_acc

### Fully connected neural network

In [ ]:
# Create model instance
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(in_features=3*256*256, out_features=128),
    nn.ReLU(),
    nn.Linear(in_features=128, out_features=256),
    nn.ReLU(),
    nn.Linear(in_features=256, out_features=64),
    nn.ReLU(),
    nn.Linear(in_features=64, out_features=num_classes),
)


# Test the model with a random batch (optional)
rand_input = torch.randn(32, 3, 256, 256)
rand_out = model(rand_input)
print(rand_out.shape)

# breakpoint()

# GPU device check - use GPU if available (CPU otherwise)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Move model to GPU (if available)
model.to(device)

# Prepare a loss function
loss_func = nn.CrossEntropyLoss()

# Create new optimizer instance
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Define total number of epochs
num_epochs = 50

# Calculate the initial model loss and accuracy
test_loss, test_acc = test(model, loss_func, test_loader, device)
print(f"Initial test accuracy: {test_acc:.4f}, test loss: {test_loss:.4f}")

# Prepare lists for accuracies and losses tracking
all_train_losses = []
all_train_acc = []
all_test_losses = []
all_test_acc = []

# Start training
for epoch in range(num_epochs):
    preds = []
    losses = []
    gt = []

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(inputs)
        loss = loss_func(logits, labels)
        loss.backward()
        optimizer.step()

        p = torch.argmax(logits, dim=1)
        preds.extend(p.cpu().numpy())
        losses.append(loss.item())
        gt.extend(labels.detach().cpu().numpy())

    test_loss, test_acc = test(model, loss_func, test_loader, device)
    train_loss = sum(losses) / len(losses)
    train_acc = accuracy_score(gt, preds)

    print(f"Epoch: {epoch}, train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, test loss: {test_loss:.4f}, test_acc: {test_acc:.4f}")

    all_train_losses.append(train_loss)
    all_train_acc.append(train_acc)
    all_test_losses.append(test_loss)
    all_test_acc.append(test_acc)

epochs = list(range(len(all_train_losses)))

plt.figure(figsize=(12, 5))

# Loss plot
plt.subplot(1, 2, 1)
sns.lineplot(x=epochs, y=all_train_losses, label='Train Loss')
sns.lineplot(x=epochs, y=all_test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Testing Loss')

# Accuracy plot
plt.subplot(1, 2, 2)
sns.lineplot(x=epochs, y=all_train_acc, label='Train Accuracy')
sns.lineplot(x=epochs, y=all_test_acc, label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Testing Accuracy')

plt.tight_layout()
plt.show()

### Define a custom neural network model

In [ ]:
class MyCustomModel(nn.Module):
    def __init__(self, num_classes, dropout=0.2):
        super().__init__()

        self.cnn_backbone = nn.Sequential(
            # Layer 1
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3),
            nn.MaxPool2d(2),
            nn.ReLU(),
            nn.Dropout(dropout),

            # Layer 2
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
            nn.MaxPool2d(2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3),
            nn.MaxPool2d(2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3),
            nn.MaxPool2d(2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.AdaptiveAvgPool2d((2, 2)),
            # Vectorize 2D feature maps
            nn.Flatten(),
        )

        self.classifier_head = nn.Sequential(
            nn.Linear(in_features=256*2*2, out_features=256),
            nn.ReLU(),
            nn.Linear(in_features=256, out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=num_classes)
        )

    def forward(self, x):
        x = self.cnn_backbone(x)
        # breakpoint()
        # return x
        return self.classifier_head(x)

## Custom CNN model training (from scratch)

In [ ]:
# Create model instance
model = MyCustomModel(num_classes)

# Test the model with a random batch (optional)
rand_input = torch.randn((32, 3, 256, 256))
rand_out = model(rand_input)
print(rand_out.shape)

# GPU device check - use GPU if available (CPU otherwise)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Move model to GPU (if available)
model.to(device)

# Prepare a loss function
loss_func = nn.CrossEntropyLoss()

# Create new optimizer instance
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Define total number of epochs
num_epochs = 50

# Calculate the initial model loss and accuracy
test_loss, test_acc = test(model, loss_func, test_loader, device)
print(f"Initial test accuracy: {test_acc:.4f}, test loss: {test_loss:.4f}")

# Prepare lists for accuracies and losses tracking
all_train_losses = []
all_train_acc = []
all_test_losses = []
all_test_acc = []

# Start training
for epoch in range(num_epochs):
    preds = []
    losses = []
    gt = []

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(inputs)
        loss = loss_func(logits, labels)
        loss.backward()
        optimizer.step()

        p = torch.argmax(logits, dim=1)
        preds.extend(p.cpu().numpy())
        losses.append(loss.item())
        gt.extend(labels.detach().cpu().numpy())

    test_loss, test_acc = test(model, loss_func, test_loader, device)
    train_loss = sum(losses) / len(losses)
    train_acc = accuracy_score(gt, preds)

    print(f"Epoch: {epoch}, train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, test loss: {test_loss:.4f}, test_acc: {test_acc:.4f}")

    all_train_losses.append(train_loss)
    all_train_acc.append(train_acc)
    all_test_losses.append(test_loss)
    all_test_acc.append(test_acc)

epochs = list(range(len(all_train_losses)))

plt.figure(figsize=(12, 5))

# Loss plot
plt.subplot(1, 2, 1)
sns.lineplot(x=epochs, y=all_train_losses, label='Train Loss')
sns.lineplot(x=epochs, y=all_test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Testing Loss')

# Accuracy plot
plt.subplot(1, 2, 2)
sns.lineplot(x=epochs, y=all_train_acc, label='Train Accuracy')
sns.lineplot(x=epochs, y=all_test_acc, label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Testing Accuracy')

plt.tight_layout()
plt.show()

### Save model weights

In [ ]:
torch.save(model.state_dict(), "model_weights.pth")

### Visualize feature maps

In [ ]:
activations = []

def save_activation_hook(module, input, output):
    activations.append(output.cpu().detach().numpy())

print(model)

model.cnn_backbone[4].register_forward_hook(save_activation_hook)
img, label = train_dataset[1]

img = img.unsqueeze(0) # Add batch dimension
img = img.to(device)
out = model(img)

plt.imshow(activations[0][0][0])
plt.show()

print(activations[0][0].shape)


## Transfer learning

### Prepare a pretrained model

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
print(model)

### Update the classifier head

In [ ]:
head_in_features = model.fc.in_features
model.fc = nn.Linear(head_in_features, num_classes)
print(model)

In [ ]:
torch.manual_seed(123)

# Test the model with a random batch (optional)
rand_input = torch.randn((32, 3, 256, 256))
rand_out = model(rand_input)
print(rand_out.shape)

# GPU device check - use GPU if available (CPU otherwise)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Move model to GPU (if available)
model.to(device)

# Prepare a loss function
loss_func = nn.CrossEntropyLoss()

# Create new optimizer instance
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001) # Lower learning rate is more suitable for transfer learning

# Define total number of epochs
num_epochs = 10

# Calculate the initial model loss and accuracy
test_loss, test_acc = test(model, loss_func, test_loader, device)
print(f"Pretrained model test accuracy: {test_acc:.4f}, test loss: {test_loss:.4f}")

# Prepare lists for accuracies and losses tracking
all_train_losses = []
all_train_acc = []
all_test_losses = []
all_test_acc = []

# Start training
for epoch in range(num_epochs):
    preds = []
    losses = []
    gt = []

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(inputs)
        loss = loss_func(logits, labels)
        loss.backward()
        optimizer.step()

        p = torch.argmax(logits, dim=1)
        preds.extend(p.cpu().numpy())
        losses.append(loss.item())
        gt.extend(labels.detach().cpu().numpy())

    test_loss, test_acc = test(model, loss_func, test_loader, device)
    train_loss = sum(losses) / len(losses)
    train_acc = accuracy_score(gt, preds)

    print(f"Epoch: {epoch}, train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, test loss: {test_loss:.4f}, test_acc: {test_acc:.4f}")

    all_train_losses.append(train_loss)
    all_train_acc.append(train_acc)
    all_test_losses.append(test_loss)
    all_test_acc.append(test_acc)

epochs = list(range(len(all_train_losses)))

plt.figure(figsize=(12, 5))

# Loss plot
plt.subplot(1, 2, 1)
sns.lineplot(x=epochs, y=all_train_losses, label='Train Loss')
sns.lineplot(x=epochs, y=all_test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Testing Loss')

# Accuracy plot
plt.subplot(1, 2, 2)
sns.lineplot(x=epochs, y=all_train_acc, label='Train Accuracy')
sns.lineplot(x=epochs, y=all_test_acc, label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Testing Accuracy')

plt.tight_layout()
plt.show()

#### Freeze the pretrained model, train only the classifier head

In [ ]:
torch.manual_seed(123)

# Initialize a new pretrained model
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# TODO: Update the classifier head (note: newly added layers are trainable by default)
head_in_features = model....
model.fc = ...

# Move model to GPU (if available)
model.to(device)

# Prepare a loss function
loss_func = nn.CrossEntropyLoss()

# Create new optimizer instance
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001) # Lower learning rate is more suitable for transfer learning

# Define total number of epochs
num_epochs = 20

# Calculate the initial model loss and accuracy
test_loss, test_acc = test(model, loss_func, test_loader, device)
print(f"Pretrained model test accuracy: {test_acc:.4f}, test loss: {test_loss:.4f}")

# Prepare lists for accuracies and losses tracking
all_train_losses = []
all_train_acc = []
all_test_losses = []
all_test_acc = []

# Start training
for epoch in range(num_epochs):
    preds = []
    losses = []
    gt = []

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(inputs)
        loss = loss_func(logits, labels)
        loss.backward()
        optimizer.step()

        p = torch.argmax(logits, dim=1)
        preds.extend(p.cpu().numpy())
        losses.append(loss.item())
        gt.extend(labels.detach().cpu().numpy())

    test_loss, test_acc = test(model, loss_func, test_loader, device)
    train_loss = sum(losses) / len(losses)
    train_acc = accuracy_score(gt, preds)

    print(f"Epoch: {epoch}, train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, test loss: {test_loss:.4f}, test_acc: {test_acc:.4f}")

    all_train_losses.append(train_loss)
    all_train_acc.append(train_acc)
    all_test_losses.append(test_loss)
    all_test_acc.append(test_acc)

epochs = list(range(len(all_train_losses)))

plt.figure(figsize=(12, 5))

# Loss plot
plt.subplot(1, 2, 1)
sns.lineplot(x=epochs, y=all_train_losses, label='Train Loss')
sns.lineplot(x=epochs, y=all_test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Testing Loss')

# Accuracy plot
plt.subplot(1, 2, 2)
sns.lineplot(x=epochs, y=all_train_acc, label='Train Accuracy')
sns.lineplot(x=epochs, y=all_test_acc, label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Testing Accuracy')

plt.tight_layout()
plt.show()


### Test model predictions

In [ ]:
model.eval()

# Model predictions
fig, axes = plt.subplots(3, 3, figsize=(6, 6))
fig.suptitle("Model predictions", fontsize=14)

for i in range(9):
    image, label = test_dataset[i]
    image = image.to(device)

    pred_prob = model(image.unsqueeze(0))
    pred_class = torch.argmax(pred_prob, dim=1)

    image = image.cpu().squeeze().permute(1, 2, 0).numpy()

    # Normalize to 0-1 range (this is optional and offers better visualization, pyplot expects the 0-1 range for float dtypes)
    image = image - np.min(image)
    image = image / np.max(image)

    ax = axes[i // 3, i % 3]
    ax.imshow(image)
    ax.set_title(f"Label: {label}, prediction: {pred_class.cpu().numpy()[0]}")
    ax.axis('off')

plt.tight_layout()
plt.show()

## Exercises



1. Experiment with different training image transforms (e.g.: rotations, noise, etc.)
2. Set the dropout to 50%
3. Modify the network layers
4. Save the trained model weights
5. Experiment with different optimizers
6. Add batch normalization to the custom CNN and the fully connected models